In [0]:
!pip install GEOparse

In [0]:
import GEOparse
import pandas as pd

def save_geo_sample_metadata(gse_id: str, table_path: str):
    print(f"📥 Downloading metadata for {gse_id}...")
    gse = GEOparse.get_GEO(geo=gse_id, destdir="/tmp", silent=True)

    # Extract and flatten metadata for each GSM sample
    records = []
    for gsm_id, gsm in gse.gsms.items():
        meta = gsm.metadata
        flat_meta = {k: v[0] if isinstance(v, list) else v for k, v in meta.items()}
        flat_meta["gsm_id"] = gsm_id
        flat_meta["gse_id"] = gse_id
        records.append(flat_meta)

    df = pd.DataFrame(records)

    # Convert to Spark and save as Delta table
    spark_df = spark.createDataFrame(df)
    spark_df.write.mode("overwrite").format("delta").saveAsTable(table_path)
    print(f"✅ Saved table: {table_path}")

# Save GSE213478 sample metadata
save_geo_sample_metadata("GSE213478", "bronze.methylation.GSE213478_samples")

# Save GSE289137 sample metadata
save_geo_sample_metadata("GSE289137", "bronze.methylation.GSE289137_samples")


In [0]:
%sql

SELEcT * FROM bronze.methylation.GSE213478_samples

In [0]:
%sql
SELEcT * FROM bronze.methylation.GSE289137_samples

## Create Merged table in Silver

In [0]:
# Load both sample tables
df1 = spark.table("bronze.methylation.GSE213478_samples")
df2 = spark.table("bronze.methylation.GSE289137_samples")

# Get column sets
cols1 = set(df1.columns)
cols2 = set(df2.columns)

# Find common columns
common_cols = list(cols1 & cols2)
print(f"✅ Common columns ({len(common_cols)}):", common_cols)

# Select only those columns
df1_common = df1.select(common_cols)
df2_common = df2.select(common_cols)

# Union and write to silver table
df_union = df1_common.unionByName(df2_common)

df_union.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver.methylation.samples")

print("✅ Saved unified sample metadata to: silver.methylation.samples")


In [0]:
%sql

SELECT * FROM silver.methylation.samples